# 🔬 Arm 3 (Priority 3 — Structural Syntax): AST-Guided Policy Optimization (AST-RL)
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Literature:** TreeDiff (ASE 2025) · VeriSeek (ICSE 2025)  

---

## 🎯 Core Concept
Standard RLVR with binary rewards ($R \in \{0, 1\}$) can still suffer from **lexical overfitting**: a model memorizes variable names and code surface without internalizing the control-flow logic.

**AST-RL** integrates the Python **Abstract Syntax Tree (AST)** into the reward loop:
$$\mathcal{R}_{\text{AST}}(y) = \exp \left( - \alpha \cdot \text{TreeDist}(\text{AST}(y), \text{AST}(y^*)) \right)$$

| Advantage | Limitation |
|---|---|
| Prevents lexical surface overfitting | Misses high-level semantic reframing |
| Structural correctness even with renamed vars | Doesn't capture narrative or creative context |

> See Table 1 §3.4 in the research proposal for the full taxonomy comparison.

In [ ]:
import os
import sys
import ast
import numpy as np
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("."))

print("✅ Arm 3 (AST-RL) Environment Initialized.")

---
## 1. AST Normalizer & Structural Similarity Engine (`simAST`)

In [ ]:
class ASTNormalizer(ast.NodeTransformer):
    """Strips variable names to expose pure control-flow structure."""
    def visit_Name(self, node):
        return ast.copy_location(ast.Name(id='_v', ctx=node.ctx), node)
    def visit_arg(self, node):
        return ast.copy_location(ast.arg(arg='_a', annotation=None), node)


def get_ast_signature(code: str) -> list:
    """Returns depth-first sequence of normalized AST node types."""
    try:
        tree = ASTNormalizer().visit(ast.parse(code))
        return [type(n).__name__ for n in ast.walk(tree)]
    except SyntaxError:
        return ["SyntaxError"]


def simAST(code_gen: str, code_ref: str) -> float:
    """Structural AST similarity in [0, 1]. Equivalent to TreeDiff/VeriSeek simAST metric."""
    sig_gen = get_ast_signature(code_gen)
    sig_ref = get_ast_signature(code_ref)
    if "SyntaxError" in sig_gen or "SyntaxError" in sig_ref:
        return 0.0
    set_gen, set_ref = set(sig_gen), set(sig_ref)
    jaccard = len(set_gen & set_ref) / max(len(set_gen | set_ref), 1)
    len_ratio = min(len(sig_gen), len(sig_ref)) / max(len(sig_gen), len(sig_ref), 1)
    return round(0.6 * jaccard + 0.4 * len_ratio, 4)


def ast_reward(code_gen: str, code_ref: str, alpha: float = 0.05) -> float:
    """TreeDiff / VeriSeek AST reward: R_AST = exp(-alpha * TreeDist(AST(y), AST(y*)))"""
    sim = simAST(code_gen, code_ref)
    tree_dist = 1.0 - sim  # approx distance
    return round(float(np.exp(-alpha * tree_dist * 100)), 4)


print("✅ ASTNormalizer, simAST, and ast_reward defined.")

---
## 2. Invariance Simulation: Same Logic, Different Surface

In [ ]:
ref_code = """
def filter_evens(numbers):
    result = []
    for n in numbers:
        if n % 2 == 0:
            result.append(n)
    return result
"""

# Same algorithm, completely different variable names — AST should be near-identical
renamed_code = """
def get_even_items(elements):
    collected = []
    for item in elements:
        if item % 2 == 0:
            collected.append(item)
    return collected
"""

# Completely different algorithm (quicksort) — AST should differ sharply
decoy_code = """
def sort_items(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    left  = [x for x in arr[1:] if x < pivot]
    right = [x for x in arr[1:] if x >= pivot]
    return sort_items(left) + [pivot] + sort_items(right)
"""

sim_renamed = simAST(renamed_code, ref_code)
sim_decoy   = simAST(decoy_code, ref_code)
r_renamed   = ast_reward(renamed_code, ref_code)
r_decoy     = ast_reward(decoy_code, ref_code)

print(f"simAST (Renamed vs Ref) : {sim_renamed * 100:.1f}%  → R_AST = {r_renamed:.3f}")
print(f"simAST (Decoy vs Ref)   : {sim_decoy * 100:.1f}%  → R_AST = {r_decoy:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=140)
# Similarity bar
axes[0].bar(['Renamed\n(Valid Logic)', 'Decoy\n(Wrong Logic)'], [sim_renamed, sim_decoy],
            color=['#28A745', '#DC3545'], edgecolor='black')
axes[0].set_title('simAST Similarity Score', fontweight='bold')
axes[0].set_ylim(0, 1.15)
# Reward bar
axes[1].bar(['Renamed\n(Valid Logic)', 'Decoy\n(Wrong Logic)'], [r_renamed, r_decoy],
            color=['#28A745', '#DC3545'], edgecolor='black')
axes[1].set_title('AST Reward: R_AST = exp(-α·TreeDist)', fontweight='bold')
axes[1].set_ylim(0, 1.15)
for ax in axes:
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/arm3_ast_similarity.png', dpi=140)
plt.show()
print("✅ AST-RL reward correctly rewards structural equivalence regardless of variable names!")